In [ ]:
using Pkg
Pkg.activate("/Users/bursche/Documents/GitHub/JPEC_BCRIT")
Base.active_project()

In [ ]:
using GeneralizedPerturbedEquilibrium
using GeneralizedPerturbedEquilibrium: Analysis
using GeneralizedPerturbedEquilibrium: InnerLayer
using GeneralizedPerturbedEquilibrium.InnerLayer: solve_inner
using GeneralizedPerturbedEquilibrium: Tearing


In [ ]:
using Plots
default(
    fontfamily="Georgia",
    margin=12Plots.mm,
    size=(800, 500),
    dpi=150
)

In [ ]:
struct TorqueBalance
    model
    params
    Q0::Float64
    P::Float64
    delta_n_p::Float64
    lu::Float64
    sval::Float64
end

function torque_balance_value(tb::TorqueBalance, Q::Number)
    Δ = solve_inner(tb.model, tb.params, ComplexF64(Q)).tearing
    jxb = -imag(1.0 / (Δ + tb.delta_n_p))
    return 2.0 * tb.P * (tb.Q0 - Q) / jxb, Δ
end

function torque_balance_scan(tb; Qmin=-10.0, Qmax=10.0, n=200)
    Qs = range(Qmin, Qmax; length=n)
    torque_out = [torque_balance_value(tb, q) for q in Qs]
    bal = [x[1] for x in torque_out]
    Δs = [x[2] for x in torque_out]
    #bal = [torque_balance_value(tb, q) for q in Qs]
    positive = isfinite.(bal) .& (bal .> 0.0)

    if !any(positive)
        return Qs, bal, NaN, 0.0
    end
    i = argmax(bal)
    Qpeak_ind = i
    Qs_positive = Qs[positive]
    bal_positive = bal[positive]

    i = argmax(bal_positive)
    Qpeak = Qs_positive[i]
    
    maxbal = bal_positive[i]

    br_crit = sqrt(maxbal / tb.lu * (tb.sval^2 / 2.0))
    return Qs, bal, Qs_positive, bal_positive, Qpeak, br_crit, Qpeak_ind, Δs
end

In [ ]:
# You can substitute any valid SLAYERParameters you already have.
# This is just an example skeleton.
p = GeneralizedPerturbedEquilibrium.InnerLayer.slayer_parameters(
    n_e=1e19, t_e=1e3, t_i=1e3,
    omega=0.0, omega_e=4, omega_i=-2,
    qval=2.0, sval_r=0.5, bt=2.0, rs=1.0, R0=3.0, mu_i=2.0, zeff=1.0,
    chi_perp=1.0, chi_tor=1.0, m=2, n=1
)
p2 = GeneralizedPerturbedEquilibrium.InnerLayer.SLAYERParameters(
    ising=p.ising,
    m=p.m, n=p.n,
    tau=p.tau, lu=p.lu, c_beta=p.c_beta, D_norm=p.D_norm,
    P_perp=p.P_perp, P_tor=p.P_tor,
    Q_e=4 / p.tauk, Q_i=-10 / p.tauk, iota_e=p.iota_e,
    tauk=p.tauk, tau_r=p.tau_r, delta_n=p.delta_n,
    rs=p.rs, R0=p.R0, bt=p.bt, sval_r=p.sval_r,
    dr_val=p.dr_val, dgeo_val=p.dgeo_val,
    eta=p.eta, d_beta=p.d_beta,
    dc_tmp=p.dc_tmp, dc_type=p.dc_type
)

#p = p2

tb = TorqueBalance(
    GeneralizedPerturbedEquilibrium.InnerLayer.SLAYERModel(;),
    p,
    0.5,
    1.0,
    1e-2,
    p.lu,
    p.sval_r
)

In [ ]:
"""

tau_h=R0*(mu0*rho)**0.5/(nns*sval*bt) ! alfven time across surface
lu=tau_r/tau_h                   ! Lundquist number
Qconv=lu**(1.0/3.0)*tau_h        ! conversion to Qs based on Cole

! note Q depends on Qconv even if omega is fixed.
Q=Qconv*omega
Q_e=-Qconv*omega_e
Q_i=-Qconv*omega_i

"""
2=-lu**(1.0/3.0)*tau_h *OE
2 = tauk *OE

In [ ]:
Qs = range(-5.0, 5.0, length=200)

vals = [torque_balance_value(tb, q) for q in Qs]

In [ ]:
"""
n=2000:
Qpeak = 0.24512256128064033
maxbal = 0.35873684252530247
br_crit = 4.668551985584323e-5

n=200 (def):
Qpeak = 0.25125628140703515
maxbal = 0.3585219071071706
br_crit = 4.667153206033017e-5
"""
Qs, bal, Qs_positive, bal_positive, Qpeak, brcrit, Qpeak_ind, Δs = torque_balance_scan(tb)

println("Qpeak = ", Qpeak)
println("maxbal = ", maximum(bal_positive))
println("maxbal = ", maximum(bal))
println("br_crit = ", brcrit)

p1 = plot(Qs, bal, lw=2, label="balance")
vline!([Qpeak], label="peak Q", linestyle=:dash)
hline!([0.0], label="zero", linestyle=:dashdot)
xlabel!("Q")
ylabel!("balance")
title!("Torque-balance scan")

p2 = plot(Qs_positive, bal_positive, lw=2, label="balance (positive)")
vline!([Qpeak], label="peak Q", linestyle=:dash)
hline!([0.0], label="zero", linestyle=:dashdot)
xlabel!("Q")
ylabel!("balance")
title!("Torque-balance scan")

plot(p1, p2, layout=(2,1), size=(800, 1000))


In [ ]:
Qpk = Qs_positive[argmax(bal_positive)]
j   = findfirst(isequal(Qpk), Qs)

lhs = 2*tb.P*(tb.Q0 - Qpk) / jxbs[j]
br_ratio = sqrt(lhs / (tb.lu * tb.sval^2 / 2))

rhs = (tb.lu * tb.sval^2 / 2) * br_ratio^2

println("lhs = ", lhs)
println("rhs = ", rhs)
println("lhs/rhs = ", lhs / rhs)

In [ ]:
#Δs = [solve_inner(tb.model, tb.params, ComplexF64(q)).tearing for q in Qs]
jxbs = [-imag(1.0 / (d + tb.delta_n_p)) for d in Δs]
T_VISC = [2.0 * tb.P * (tb.Q0 - q) for (q, jxb) in zip(Qs, jxbs)]
br_t = brcrit
println(tb.lu, " ", tb.sval, " ", br_t, " ", p.bt, " ", tb.delta_n_p)
T_EM = tb.lu * tb.sval^2/2 * (br_t / p.bt)^2 * jxbs
i = Qpeak_ind
println(T_VISC[i]/(tb.lu * tb.sval^2/2 * (brcrit)^2 * jxbs[i]))
println(T_VISC[i] / (tb.lu * tb.sval^2/2 * (brcrit * p.bt)^2 * jxbs[i]))
println(T_VISC[i] / ( (brcrit / p.bt)^2 * jxbs[i]))

#T_EM ∝ S ξ̂ (br/Bφ)² Im[-Δ̂(Q)⁻¹]
#T_visc ∝ 2 P (Q0 − Q)

p1 = plot(Qs, imag.(Δs), label="Im(Δ)", lw=2)
#plot!(p1, Qs, real.(Δs), label="Re(Δ)", lw=2)
xlabel!(p1, "Q")
ylabel!(p1, "Δ")
title!(p1, "Inner-layer Δ(Q)")

p2 = plot(Qs, jxbs, label="jxb", lw=2)
plot!(p2, Qs, T_VISC, label="2P(Q0-Q)", lw=2)
plot!(p2, Qs, T_EM, label="T_EM", lw=2)
xlabel!(p2, "Q")
ylabel!(p2, "jxb")
title!(p2, "jxb(Q) = -Im[1/(Δ + δ_n_p)]")

p3 = plot(Qs, real.(bal), label="Re(balance)", lw=2)
plot!(p3, Qs, imag.(bal), label="Im(balance)", lw=2)
xlabel!(p3, "Q")
ylabel!(p3, "balance")
title!(p3, "2P(Q0-Q)/jxb")

plot(p1, p2, p3, layout=(3,1), size=(800, 1000))